# Build the unified genome-wide variant panel (Google Batch / dsub)

One whole-cohort panel, every AoU sample with ACAF coverage -- no `--keep`, no `SAMPLE_SET`, no population-specific MAF/HWE/geno QC (that stays population-specific, deferred to `03_genome_wide_qc_thinning_batch_submit.ipynb`'s per-`SAMPLE_SET` step). Per chromosome: HM3-agreeing variants (real ID+REF+ALT harmonization against 1000G, not a bare ID match) union a randomly `--thin`ned genome-wide set for GRM density, then a lenient *pooled* (not population-specific) `--maf 0.001 --geno 0.05` filter -- cuts dead-weight variants that would never survive any `SAMPLE_SET`'s own QC anyway (a variant near-monomorphic pooled across all 535k samples is near-monomorphic in every subgroup too) and drops variants with unreliable genotyping, without pre-judging the real population-specific MAF/HWE decision that still happens downstream.

Exists so every `SAMPLE_SET` restricts from the *same* starting variant list -- previously each `SAMPLE_SET` independently random-`--thin`ned from scratch, so different `SAMPLE_SET`s ended up with genuinely different variant IDs even where both would otherwise have kept a variant, which blocked anything needing to compare/combine across them.

No bed export here -- at ~535k samples this pgen alone lands around several hundred GB even after the pooled filter, and PLINK 1.9's 2-bits/genotype bed format would be ~2x that with no compression. Bed export happens once, downstream, on each `SAMPLE_SET`'s own much smaller `--keep`+QC'd panel (`03`'s `qcb-submit` already does this).

Dsub/Batch only (`--image gcr.io/google.com/cloudsdktool/cloud-sdk:slim` + `gsutil -u "$PROJECT_ID"` for this Requester Pays bucket).

## Prerequisites

`dsub` installed, `gcloud` authenticated, `explore_1kg_reference.ipynb` already run (for the HM3 harmonization table).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Same project/bucket values as `03_genome_wide_qc_thinning_batch_submit.ipynb`. `KG_OUT_PREFIX`'s `1kg_all_qc.acount` (ID/REF/ALT for every HM3 QC'd variant) is what harmonization runs against. `THIN_P` reused from prior chr22 calibration -- same value the old per-`SAMPLE_SET` thinning used.

In [ ]:
import os

PROJECT_ID = "wb-swift-sprout-7231"
REGION = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
WORKSPACE_BUCKET_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning (selecting the ACAF data source
# below, and filename/log tags) throughout this notebook. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "phenotypic_covariance_v9"

ACAF_BUCKET_GS = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
ANCESTRY_BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/{PROJECT_DIR}/01_ancestry_filtering"

KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # explore_1kg_reference.ipynb's output, CDR-independent
KG_OUT_PREFIX = f"{KG_DIR}/1kg_all_qc"
assert os.path.isfile(f"{KG_OUT_PREFIX}.acount"), f"missing {KG_OUT_PREFIX}.acount -- run explore_1kg_reference.ipynb first"

# single whole-cohort panel -- no SAMPLE_SET, no --keep
BUCKET_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/unified_panel"
PLINK_BIN_GS = f"{BUCKET_DIR_GS}/bin/plink2"
KG_HARMONIZE_GS = f"{BUCKET_DIR_GS}/1kg_id_ref_alt.sorted"

THIN_P = 0.1   # lowered from the old per-SAMPLE_SET thinning's 0.2 -- storage/compute cost concerns on
# a genome-wide, whole-cohort (535k sample) panel; drop to 0.05 for a further ~2x cut if still too large

# per-task machine -- same sizing 03 (formerly 02) landed on after chr1-4's real
# OOM/disk failure on a smaller machine (n1-standard-8, --disk-size 100)
MACHINE_VCPUS = 16
MEMORY_MB = 55000

print(ACAF_BUCKET_GS)
print(BUCKET_DIR_GS)

## Build and stage the 1000G ID+REF+ALT harmonization table

Sorted `ID REF ALT` from `1kg_all_qc.acount` -> staged to the bucket (not Requester Pays) for `--input`. Each per-chromosome task below does its own `comm -12` against this after relabeling ACAF's IDs -- true harmonization, not a bare ID match.

In [ ]:
%%bash -s "$KG_OUT_PREFIX" "$KG_HARMONIZE_GS"
set -e
KG_OUT_PREFIX=$1
KG_HARMONIZE_GS=$2

LOCAL_TABLE="/tmp/1kg_id_ref_alt.sorted"
awk 'NR>1 {print $2, $3, $4}' "${KG_OUT_PREFIX}.acount" | LC_ALL=C sort > "$LOCAL_TABLE"
wc -l "$LOCAL_TABLE"

gcloud storage cp "$LOCAL_TABLE" "$KG_HARMONIZE_GS"
gcloud storage ls -l "$KG_HARMONIZE_GS"

## Stage the plink2 binary (one-time)

No `wget`/`curl` on the default image -- stage once, localize via `--input`.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

"$BIN_DIR/plink2" --version

In [ ]:
%%bash -s "$PLINK_BIN_GS"
set -e
PLINK_BIN_GS=$1

local_plink="$HOME/bin/plink2"
if [ ! -x "$local_plink" ]; then
  echo "no local plink2 at $local_plink -- run the cell above first" >&2
  exit 1
fi

gcloud storage cp "$local_plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS"

## Stage 1 -- single-chromosome validation

chr22. Same ID-relabel/harmonize prep `explore_hm3_ancestry_panel.ipynb` already uses, plus a randomly-`--thin`ned candidate set unioned in for GRM density, then a lenient pooled `--maf 0.001 --geno 0.05` filter (not population-specific QC -- that's still deferred to `03`'s per-`SAMPLE_SET` step).

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS" "$PLINK_BIN_GS" "$KG_HARMONIZE_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8
PLINK_BIN_GS=$9
KG_HARMONIZE_GS=${10}
BUCKET_DIR_GS=${11}
THIN_P=${12}
MACHINE_VCPUS=${13}
MEMORY_MB=${14}
ANCESTRY_BUCKET_DIR_GS=${15}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"
CHR=22
OUT_NAME="chr${CHR}_unified_${CDR_VERSION}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "unified-panel-chr22-validation" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --input KG_HARMONIZE="$KG_HARMONIZE_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env CHR_PGEN_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pgen" \
  --env CHR_PVAR_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pvar" \
  --env CHR_PSAM_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.psam" \
  --env OUT_NAME="$OUT_NAME" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --make-pgen \
      --out chr_relabeled

    grep -v "^##" chr_relabeled.pvar | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > chr_id_ref_alt.sorted
    LC_ALL=C comm -12 chr_id_ref_alt.sorted "$KG_HARMONIZE" | awk "{print \$1}" > hm3_snps.ids
    echo "HM3-agreeing: $(wc -l < hm3_snps.ids)"

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --thin '"$THIN_P"' \
      --write-snplist \
      --out chr_thinned_candidate
    mv chr_thinned_candidate.snplist thinned_snps.ids
    echo "Randomly thinned: $(wc -l < thinned_snps.ids)"

    LC_ALL=C sort -u hm3_snps.ids thinned_snps.ids > union_snps.ids
    echo "Union (HM3 + thinned): $(wc -l < union_snps.ids)"

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --extract union_snps.ids \
      --maf 0.001 \
      --geno 0.05 \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  > /tmp/unified_validation_job_id.txt

cat /tmp/unified_validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
JOB_ID=$(tail -1 /tmp/unified_validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

echo
echo "--- output files ---"
gcloud storage ls -l "${BUCKET_DIR_GS}/" 2>/dev/null | grep chr22 || echo "(none yet -- check task status above)"

## Stage 2 -- full run (all 22 chromosomes)

Run after Stage 1 succeeds. Same `--tasks` TSV pattern as `03`.

In [ ]:
CHR_LENGTHS = {
    1: 248_956_422, 2: 242_193_529, 3: 198_295_559, 4: 190_214_555,
    5: 181_538_259, 6: 170_805_979, 7: 159_345_973, 8: 145_138_636,
    9: 138_394_717, 10: 133_797_422, 11: 135_086_622, 12: 133_275_309,
    13: 114_364_328, 14: 107_043_718, 15: 101_991_189, 16: 90_338_345,
    17: 83_257_441, 18: 80_373_285, 19: 58_617_616, 20: 64_444_167,
    21: 46_709_983, 22: 50_818_468,
}
CHRS_LARGEST_FIRST = sorted(CHR_LENGTHS, key=CHR_LENGTHS.get, reverse=True)

TASKS_PATH = "/tmp/unified_panel_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in CHRS_LARGEST_FIRST:
        out_name = f"chr{chr_num}_unified_{CDR_VERSION}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PLINK_BIN_GS" "$KG_HARMONIZE_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PLINK_BIN_GS=$8
KG_HARMONIZE_GS=$9
BUCKET_DIR_GS=${10}
THIN_P=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}
ANCESTRY_BUCKET_DIR_GS=${14}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "unified-panel-genome-wide" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --input KG_HARMONIZE="$KG_HARMONIZE_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --make-pgen \
      --out chr_relabeled

    grep -v "^##" chr_relabeled.pvar | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > chr_id_ref_alt.sorted
    LC_ALL=C comm -12 chr_id_ref_alt.sorted "$KG_HARMONIZE" | awk "{print \$1}" > hm3_snps.ids
    echo "HM3-agreeing: $(wc -l < hm3_snps.ids)"

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --thin '"$THIN_P"' \
      --write-snplist \
      --out chr_thinned_candidate
    mv chr_thinned_candidate.snplist thinned_snps.ids
    echo "Randomly thinned: $(wc -l < thinned_snps.ids)"

    LC_ALL=C sort -u hm3_snps.ids thinned_snps.ids > union_snps.ids
    echo "Union (HM3 + thinned): $(wc -l < union_snps.ids)"

    "$PLINK_BIN" \
      --pfile chr_relabeled \
      --extract union_snps.ids \
      --maf 0.001 \
      --geno 0.05 \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/unified_panel_tasks.tsv \
  > /tmp/unified_genome_wide_job_id.txt

cat /tmp/unified_genome_wide_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/unified_genome_wide_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Merge chromosomes into the unified genome-wide pfile (dsub)

`--pmerge-list`, same dsub-based merge pattern as `03_genome_wide_qc_thinning_batch_submit.ipynb`'s own merge step -- not Requester Pays (this project's own bucket), so plain `--input-recursive`/`--output-recursive` work directly. pgen only, no bed export (see intro -- deferred to each `SAMPLE_SET`'s own smaller panel downstream).

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$BUCKET_DIR_GS" "$CDR_VERSION" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
SERVICE_ACCOUNT=$3
NETWORK=$4
SUBNETWORK=$5
BUCKET_DIR_GS=$6
CDR_VERSION=$7
MACHINE_VCPUS=$8
MEMORY_MB=$9
ANCESTRY_BUCKET_DIR_GS=${10}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"
MERGED_NAME="unified_panel_${CDR_VERSION}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "unified-panel-merge" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input-recursive PANEL_DIR="$BUCKET_DIR_GS" \
  --env CDR_VERSION="$CDR_VERSION" \
  --env MERGED_NAME="$MERGED_NAME" \
  --env MACHINE_VCPUS="$MACHINE_VCPUS" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    PLINK_BIN="${PANEL_DIR}/bin/plink2"
    chmod +x "$PLINK_BIN"

    MERGE_LIST_PATH="merge_list.txt"
    > "$MERGE_LIST_PATH"
    for chr_num in $(seq 1 22); do
      echo "${PANEL_DIR}/chr${chr_num}_unified_${CDR_VERSION}" >> "$MERGE_LIST_PATH"
    done

    "$PLINK_BIN" \
      --pmerge-list "$MERGE_LIST_PATH" \
      --make-pgen \
      --threads "$MACHINE_VCPUS" \
      --out "${OUT_DIR}/${MERGED_NAME}"

    echo "Unified panel variant count:"
    grep -vc "^##" "${OUT_DIR}/${MERGED_NAME}.pvar"
    echo "Sample count:"
    wc -l < "${OUT_DIR}/${MERGED_NAME}.psam"
  ' \
  > /tmp/unified_merge_job_id.txt

cat /tmp/unified_merge_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/unified_merge_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Next steps

Run `03_genome_wide_qc_thinning_batch_submit.ipynb` next, once per `SAMPLE_SET` -- it reads `unified_panel_{CDR_VERSION}` directly (`--keep` + population-specific MAF/HWE/geno QC), no more raw ACAF access needed at that step.